# 🩺 Prediksi Risiko Prediabetes Berdasarkan Faktor Gaya Hidup
## Menggunakan Model XGBoost
---
### 📋 Ringkasan Keputusan

| Item | Keputusan |
|---|---|
| **Target Label** | `Risk_Level` (Tidak Berisiko / Sedang / Tinggi) |
| **Algoritma** | XGBoost |
| **Jumlah Fitur** | 12 fitur |
| **Imbalance Handling** | `compute_sample_weight` (balanced) |

> ⚠️ **Catatan Revisi**: SMOTE dihapus dan diganti `compute_sample_weight` karena:
> - Imbalance 70:30 tidak cukup parah untuk SMOTE
> - SMOTE di luar CV fold menyebabkan **data leakage**
> - `sample_weight` lebih aman dan native di XGBoost
> - Data sudah besar (100k baris), tidak perlu data sintetis


## STEP 0 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import clone
from xgboost import XGBClassifier

from src.scoring import (
    RISK_LABELS,
    RISK_LABEL_TO_CODE,
    SCORING_THRESHOLDS,
    calculate_risk_score,
    create_risk_level,
)

import warnings
warnings.filterwarnings('ignore')

print('? Library berhasil diimport!')

## STEP 1 — Load Dataset

In [ ]:
file_path = "../data/diabetes_young_adults_india.csv"
df_risk = pd.read_csv(file_path)

print(f'Shape dataset  : {df_risk.shape}')
print(f'Jumlah baris   : {df_risk.shape[0]:,}')
print(f'Jumlah kolom   : {df_risk.shape[1]}')
df_risk.head()

## STEP 2 — Drop Fitur yang Tidak Digunakan

| Fitur | Alasan Drop |
|---|---|
| `ID` | Hanya identifier, bukan fitur prediktif |
| `Gender` | Distribusi prediabetes hampir sama antar gender |
| `Region` | Konteks geografis India, tidak relevan secara umum |
| `Family_Income` | Konteks sosioekonomik India, tidak relevan |
| `Parent_Diabetes_Type` | 65k missing value (~65%) |
| `Fast_Food_Intake` | Overlap/redundant dengan `Dietary_Habits` |
| `Cholesterol_Level` | Lebih relevan ke penyakit jantung |
| `Screen_Time` | Korelasi sangat lemah dengan diabetes |
| `Diabetes_Type` | 75% missing value |
| `Prediabetes` | Tidak digunakan — kita pakai `Risk_Level` |


In [ ]:
cols_to_drop = [
    'ID', 'Gender', 'Region', 'Family_Income',
    'Parent_Diabetes_Type', 'Fast_Food_Intake', 'Cholesterol_Level',
    'Screen_Time', 'Diabetes_Type', 'Prediabetes'
]
df_risk = df_risk.drop(columns=cols_to_drop)

print(f'Fitur setelah drop : {df_risk.shape[1]} kolom')
print(f'Kolom tersisa      : {df_risk.columns.tolist()}')

## STEP 3 — Cek Missing Value

In [ ]:
print('Missing value per kolom:')
print(df_risk.isnull().sum())
print(f'\nTotal missing value: {df_risk.isnull().sum().sum()}')

## STEP 4 — Buat Label Risk_Level (Scoring Berbasis Literatur Medis)

Skor dihitung dari faktor genetik, klinis, dan gaya hidup. Threshold mengacu pada standar klinis:
- **HbA1c**: ≥5.7% (Prediabetes), ≥6.5% (Diabetes)
- **Fasting Blood Sugar**: ≥100 mg/dL (Prediabetes), ≥126 mg/dL (Diabetes)
- **BMI**: ≥25 (Overweight), ≥30 (Obesitas)

| Skor Total | Label |
|---|---|
| ≥ 14 | Tinggi |
| 8 – 13 | Sedang |
| < 8 | Tidak Berisiko |


In [ ]:
df_risk['Risk_Score'] = df_risk.apply(calculate_risk_score, axis=1)
df_risk['Risk_Level'] = df_risk['Risk_Score'].apply(create_risk_level)

print('Distribusi Risk_Level:')
print(df_risk['Risk_Level'].value_counts())
print()
print(df_risk['Risk_Level'].value_counts(normalize=True).round(3))
print()
print('Threshold kategori risiko:')
print('  Skor < 8  -> Tidak Berisiko')
print('  Skor 8-13 -> Sedang')
print('  Skor >=14 -> Tinggi')

## STEP 5 — Encoding Fitur Kategorikal

| Fitur | Tipe Encoding | Alasan |
|---|---|---|
| `Family_History_Diabetes` | Binary (Yes=1, No=0) | Hanya 2 nilai |
| `Smoking` | Binary (Yes=1, No=0) | Hanya 2 nilai |
| `Alcohol_Consumption` | Binary (Yes=1, No=0) | Hanya 2 nilai |
| `Physical_Activity_Level` | Ordinal (Sedentary=0, Moderate=1, Active=2) | Ada urutan logis |
| `Dietary_Habits` | Ordinal (Unhealthy=0, Moderate=1, Healthy=2) | Ada urutan logis |
| `Risk_Level` | Ordinal (Tidak Berisiko=0, Sedang=1, Tinggi=2) | Ada urutan logis |


In [ ]:
# Binary encoding
binary_cols = ['Family_History_Diabetes', 'Smoking', 'Alcohol_Consumption']
for col in binary_cols:
    df_risk[col] = df_risk[col].map({'Yes': 1, 'No': 0})

# Ordinal encoding
df_risk['Physical_Activity_Level'] = df_risk['Physical_Activity_Level'].map(
    {'Sedentary': 0, 'Moderate': 1, 'Active': 2}
)
df_risk['Dietary_Habits'] = df_risk['Dietary_Habits'].map(
    {'Unhealthy': 0, 'Moderate': 1, 'Healthy': 2}
)

# Target encoding
df_risk['Risk_Level'] = df_risk['Risk_Level'].map(RISK_LABEL_TO_CODE)

print('? Encoding selesai!')
print()
print('Tipe data setelah encoding:')
print(df_risk.dtypes)

## STEP 6 — Verifikasi Dataset Final

In [ ]:
assert df_risk.isnull().sum().sum() == 0, '❌ Masih ada missing value!'

print(f'Shape final    : {df_risk.shape}')
print(f'Total missing  : {df_risk.isnull().sum().sum()}')
print('✅ Dataset siap digunakan!')
print()
print('Statistik deskriptif:')
df_risk.describe()

## STEP 6B — Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi Risk_Level
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df_risk['Risk_Level'].value_counts().sort_index()
labels = ['Tidak Berisiko', 'Sedang', 'Tinggi']
colors = ['#4C9BE8', '#F5A623', '#E85C5C']

axes[0].bar(labels, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Distribusi Target: Risk_Level', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporsi Kelas Risk_Level', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df_risk.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur numerik per kelas Risk_Level
num_cols = ['Age', 'BMI', 'HbA1c', 'Fasting_Blood_Sugar',
            'Genetic_Risk_Score', 'Sleep_Hours', 'Stress_Level']

risk_labels = {0: 'Tidak Berisiko', 1: 'Sedang', 2: 'Tinggi'}
colors = ['#4C9BE8', '#F5A623', '#E85C5C']  # biru, oranye, merah

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for cls, color in zip([0, 1, 2], colors):
        subset = df_risk[df_risk['Risk_Level'] == cls][col]
        axes[i].hist(subset, bins=30, alpha=0.5, color=color, label=risk_labels[cls])
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('Distribusi Fitur Numerik per Kelas Risk_Level',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## STEP 7 — Split Fitur dan Target

In [ ]:
FEATURES = [
    'Age', 'BMI', 'HbA1c', 'Fasting_Blood_Sugar',
    'Genetic_Risk_Score', 'Family_History_Diabetes',
    'Physical_Activity_Level', 'Dietary_Habits',
    'Smoking', 'Alcohol_Consumption',
    'Sleep_Hours', 'Stress_Level'
]
TARGET = 'Risk_Level'
FORBIDDEN_FEATURES = {'ID', 'Prediabetes', 'Diabetes_Type', 'Risk_Score', 'Risk_Level'}

leakage_features = FORBIDDEN_FEATURES.intersection(FEATURES)
if leakage_features:
    raise ValueError(f'Fitur leakage tidak boleh dipakai training: {leakage_features}')

X_risk = df_risk[FEATURES]
y_risk = df_risk[TARGET]

print(f'Shape X (fitur) : {X_risk.shape}')
print(f'Shape y (target): {y_risk.shape}')
print(f'Fitur ({len(FEATURES)}): {FEATURES}')

## STEP 8 — Train-Test Split

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_risk, y_risk,
    test_size=0.2,
    random_state=42,
    stratify=y_risk        # jaga proporsi kelas di train & test
)

print(f'X_train : {X_train_r.shape}')
print(f'X_test  : {X_test_r.shape}')
print()
print('Distribusi kelas di train:')
print(y_train_r.value_counts(normalize=True).round(3))
print()
print('Distribusi kelas di test:')
print(y_test_r.value_counts(normalize=True).round(3))

## STEP 9 — Hitung sample_weight (Pengganti SMOTE)

> **Kenapa bukan SMOTE?**
> - Imbalance 70:30 tidak cukup parah untuk butuh data sintetis
> - SMOTE yang diterapkan **sebelum** CV fold menyebabkan **data leakage**
> - `compute_sample_weight` lebih aman, native, dan tidak menambah data
>
> **Cara kerja**: Kelas minoritas diberi bobot lebih tinggi saat training,
> sehingga model tidak bias ke kelas mayoritas.


In [ ]:
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_r)

print('Distribusi bobot per kelas:')
for cls, name in zip([0, 1, 2], ['Tidak Berisiko', 'Sedang', 'Tinggi']):
    mask = y_train_r == cls
    print(f'  {name:<20}: {mask.sum():>6,} sampel | avg weight = {sample_weights[mask].mean():.4f}')

## STEP 10 — Training XGBoost

In [ ]:
model_risk = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

model_risk.fit(X_train_r, y_train_r, sample_weight=sample_weights)
print('✅ Model berhasil ditraining!')

## STEP 11 — Cross Validation (5-Fold, Bebas Leakage)

> **Perbaikan dari versi sebelumnya**: `sample_weight` dihitung **di dalam** setiap fold,
> bukan sebelum CV. Ini memastikan data validasi tidak "bocor" ke proses pembobotan.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_r, y_train_r)):
    X_fold_train = X_train_r.iloc[train_idx]
    y_fold_train = y_train_r.iloc[train_idx]
    X_fold_val   = X_train_r.iloc[val_idx]
    y_fold_val   = y_train_r.iloc[val_idx]

    # Hitung weight HANYA dari data train fold (tidak bocor ke val)
    fold_weights = compute_sample_weight(class_weight='balanced', y=y_fold_train)

    fold_model = clone(model_risk)
    fold_model.fit(X_fold_train, y_fold_train, sample_weight=fold_weights)

    score = fold_model.score(X_fold_val, y_fold_val)
    cv_scores.append(score)
    print(f'Fold {fold+1}: {score:.4f}')

cv_scores = np.array(cv_scores)
print()
print(f'Mean Accuracy : {cv_scores.mean():.4f}')
print(f'Std           : {cv_scores.std():.4f}')
print(f'Min           : {cv_scores.min():.4f}')
print(f'Max           : {cv_scores.max():.4f}')

## STEP 12 — Evaluasi Model

In [ ]:
y_pred_r = model_risk.predict(X_test_r)
classes  = ['Tidak Berisiko', 'Sedang', 'Tinggi']

# Classification Report
print('=== Classification Report ===')
print(classification_report(y_test_r, y_pred_r, target_names=classes))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm_r = confusion_matrix(y_test_r, y_pred_r)
ConfusionMatrixDisplay(confusion_matrix=cm_r, display_labels=classes).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Confusion Matrix - XGBoost Risk Level', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## STEP 13 — FPR & FNR per Kelas

> **Penting dalam konteks medis:**
> - **FNR (False Negative Rate)** = pasien berisiko yang **tidak terdeteksi** → lebih berbahaya
> - **FPR (False Positive Rate)** = pasien normal yang **salah diklasifikasi** berisiko


In [ ]:
print('=== FPR dan FNR per Kelas ===')
print(f"{'Kelas':<20} {'TP':>7} {'FP':>7} {'FN':>7} {'TN':>7} {'FPR':>8} {'FNR':>8}")
print('-' * 65)

fprs, fnrs = [], []
for i, kelas in enumerate(classes):
    TP = cm_r[i, i]
    FN = cm_r[i, :].sum() - TP
    FP = cm_r[:, i].sum() - TP
    TN = cm_r.sum() - TP - FP - FN

    FPR = FP / (FP + TN) * 100 if (FP + TN) > 0 else 0
    FNR = FN / (FN + TP) * 100 if (FN + TP) > 0 else 0

    fprs.append(FPR)
    fnrs.append(FNR)

    print(f'{kelas:<20} {TP:>7,} {FP:>7,} {FN:>7,} {TN:>7,} {FPR:>7.2f}% {FNR:>7.2f}%')

print('-' * 65)
print(f"{'Macro Avg':<20} {'':>7} {'':>7} {'':>7} {'':>7} {np.mean(fprs):>7.2f}% {np.mean(fnrs):>7.2f}%")

## STEP 14 — ROC-AUC Multiclass

In [ ]:
y_prob_r = model_risk.predict_proba(X_test_r)
roc_auc  = roc_auc_score(y_test_r, y_prob_r, multi_class='ovr', average='weighted')

print(f'ROC-AUC Score (weighted OvR): {roc_auc:.4f}')
print()
print('Interpretasi:')
print('  ≥ 0.90 : Excellent')
print('  ≥ 0.80 : Good')
print('  ≥ 0.70 : Fair')
print(f'  → Model ini: {"Excellent ✅" if roc_auc >= 0.90 else "Good ✅" if roc_auc >= 0.80 else "Fair ⚠️"}')

## STEP 15 — Feature Importance

In [ ]:
importance    = model_risk.feature_importances_
feature_names = X_risk.columns.tolist()
indices       = np.argsort(importance)[::-1]
sorted_feats  = [feature_names[i] for i in indices]
sorted_imp    = importance[indices]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(sorted_feats[::-1], sorted_imp[::-1], color='#4C9BE8', edgecolor='black')

for bar, val in zip(bars, sorted_imp[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_title('Feature Importance - XGBoost Risk Level', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_xlim(0, max(sorted_imp) + 0.05)
plt.tight_layout()
plt.show()

print()
print('Top 5 Fitur Paling Penting:')
for i in range(5):
    print(f'  {i+1}. {sorted_feats[i]:<30} {sorted_imp[i]:.4f}')

---
## ✅ Ringkasan Pipeline

| Item | Nilai |
|---|---|
| Total data | 100,000 baris |
| Fitur final | 12 fitur |
| Target | `Risk_Level` (3 kelas) |
| Training set | 80,000 baris |
| Test set | 20,000 baris |
| Missing value | 0 |
| Imbalance handling | `compute_sample_weight(balanced)` |
| Cross Validation | 5-Fold Stratified (bebas leakage) |
| Evaluasi tambahan | FPR, FNR, ROC-AUC multiclass |


# Step 16 - Save Model


In [ ]:
import joblib
import os

model_path = PROJECT_ROOT / 'models'
os.makedirs(model_path, exist_ok=True)

artifact = {
    'model': model_risk,
    'feature_columns': FEATURES,
    'target_classes': RISK_LABELS,
    'class_names': RISK_LABELS,
    'final_model_name': 'XGBClassifier Risk Level Scoring v2',
    'feature_importance': dict(zip(FEATURES, map(float, model_risk.feature_importances_))),
    'scoring_version': 'risk_scoring_v2_threshold_8_14_no_low_genetic_point',
    'scoring_thresholds': SCORING_THRESHOLDS,
    'forbidden_training_features': sorted(FORBIDDEN_FEATURES),
}

joblib.dump(artifact, model_path / 'model_xgboost_risk_level.pkl')
print('? Model dan metadata berhasil disimpan!')
print(f'File tersimpan di folder {model_path}')

model_loaded = joblib.load(model_path / 'model_xgboost_risk_level.pkl')